# MACHINE LEARNING — LAB 3
## Decision Tree Classification — Training, Evaluation & Overfitting

**Student Name:** Muhammad Umar
**Student ID:** 023-24-0260
**Section:**  H
**GitHub Profile:** https://github.com/muhammadumarofficial


## 1. Problem Definition

**Task 1–2**

The goal is to predict whether a telecom customer will churn or not based on their customer and service information. This is important for the business because identifying customers who are likely to churn can help the company take actions to retain them.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from math import log2

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


## 2. Dataset Verification

**Task 1:** Load `clean_churn.csv`, verify its shape, data types, and missing values.


In [ ]:
# ============================================================
# TASK 1 — LOAD AND VERIFY DATASET
# ============================================================

df = pd.read_csv("C:\Users\umarc\Desktop\ML-Lab Practice\clean_churn.csv")

print("First 5 rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nChurn Distribution:")
print(df["Churn"].value_counts())


**Interpretation:** The dataset shape, column types, and missing-value counts above verify whether the cleaned dataset is ready for machine learning.


## 3. Feature / Target Preparation

**Task 3–5**

- `y` is the target (`Churn`) encoded as 0/1.
- `X` contains the remaining relevant features.
- Identifier columns are removed if present.
- Remaining categorical columns are one-hot encoded.

**Why one-hot encoding?** Categorical variables can contain multiple categories, so one-hot encoding converts each category into numeric indicator columns without imposing an artificial numerical order.


In [ ]:
# ============================================================
# TASK 3 — DEFINE X AND y
# ============================================================

y = df["Churn"].copy()

if y.dtype == "object":
    y = y.map({"No": 0, "Yes": 1})

X = df.drop(columns=["Churn"]).copy()

possible_id_columns = [
    "customerID",
    "CustomerID",
    "customer_id",
    "ID",
    "id"
]

for col in possible_id_columns:
    if col in X.columns:
        X = X.drop(columns=[col])

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget values:")
print(y.value_counts())


In [ ]:
# ============================================================
# TASK 4 — CONVERT CATEGORICAL FEATURES
# ============================================================

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns

print("Categorical columns:")
print(list(categorical_columns))

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("\nX after encoding:")
display(X.head())


In [ ]:
# ============================================================
# TASK 5 — VERIFY X
# ============================================================

print("All X columns numeric:",
      X.select_dtypes(exclude=np.number).shape[1] == 0)

print("\nTotal missing values in X:",
      X.isnull().sum().sum())

print("\nX data types:")
print(X.dtypes)


**Interpretation:** `X` should contain only numeric columns and zero missing values before model training.


## 4. Train / Test Split

**Task 6–7**

An 80/20 split is used. `stratify=y` is sensible because the Churn target is imbalanced; it keeps approximately the same class proportions in training and testing data.


In [ ]:
# ============================================================
# TASK 6 — TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test split completed.")


In [ ]:
# ============================================================
# TASK 7 — REPORT SHAPES
# ============================================================

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 5. Baseline Decision Tree

**Task 8–9**

A baseline Decision Tree is trained using default settings, with only `random_state` fixed.


In [ ]:
# ============================================================
# TASK 8 — BASELINE DECISION TREE
# ============================================================

baseline_model = DecisionTreeClassifier(random_state=42)

baseline_model.fit(X_train, y_train)

y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

print("Training and testing predictions generated.")


In [ ]:
# ============================================================
# TASK 9 — TREE DEPTH
# ============================================================

print("Baseline Tree Depth:", baseline_model.get_depth())


**Interpretation:** The baseline tree is unrestricted, so scikit-learn determines its final depth based on its default stopping rules. A very deep tree can be a warning sign for overfitting.


## 6. Model Evaluation

**Task 10–12**

Accuracy alone can be misleading with class imbalance, so accuracy, precision, recall, F1-score, and the confusion matrix are reported.


In [ ]:
# ============================================================
# TASK 10 — BASELINE METRICS
# ============================================================

accuracy = accuracy_score(y_test, y_test_pred)

precision = precision_score(
    y_test, y_test_pred, zero_division=0
)

recall = recall_score(
    y_test, y_test_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_test_pred, zero_division=0
)

baseline_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

display(baseline_results)


In [ ]:
# ============================================================
# TASK 11 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.title("Baseline Decision Tree - Confusion Matrix")
plt.show()

tn, fp, fn, tp = cm.ravel()

print("Actual churners correctly caught (TP):", tp)
print("Actual churners missed (FN):", fn)


**Task 12 — Interpretation**

Because the target classes are imbalanced, **recall and F1-score** are especially useful. Recall tells us how many actual churners were correctly identified, while F1 balances precision and recall.


## 7. Overfitting Investigation

**Task 13–15**

A large difference between training and test accuracy suggests overfitting. Different `max_depth` values are tested to control tree complexity.


In [ ]:
# ============================================================
# TASK 13 — TRAINING VS TESTING ACCURACY
# ============================================================

train_accuracy = accuracy_score(
    y_train, y_train_pred
)

test_accuracy = accuracy_score(
    y_test, y_test_pred
)

print("Training Accuracy:", train_accuracy)
print("Testing Accuracy :", test_accuracy)
print("Train-Test Gap   :", train_accuracy - test_accuracy)


**Interpretation:** If training accuracy is much higher than testing accuracy, the baseline tree is likely overfitting the training data.


In [ ]:
# ============================================================
# TASK 14 — DIFFERENT max_depth VALUES
# ============================================================

depths = [2, 3, 4, 5, 6, 8, 10, None]

depth_results = []

for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    depth_results.append({
        "max_depth": (
            depth if depth is not None
            else "Unrestricted"
        ),
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "gap": train_acc - test_acc
    })

depth_results_df = pd.DataFrame(depth_results)

display(depth_results_df)


In [ ]:
# ============================================================
# TASK 15 — TRAIN VS TEST ACCURACY PLOT
# ============================================================

plot_depths = [2, 3, 4, 5, 6, 8, 10, 12]

train_scores = []
test_scores = []

for depth in plot_depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_scores.append(
        accuracy_score(
            y_train,
            model.predict(X_train)
        )
    )

    test_scores.append(
        accuracy_score(
            y_test,
            model.predict(X_test)
        )
    )

plt.figure(figsize=(9, 5))

plt.plot(
    plot_depths,
    train_scores,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    plot_depths,
    test_scores,
    marker="o",
    label="Testing Accuracy"
)

plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Training vs Testing Accuracy")
plt.legend()
plt.grid(True)
plt.show()


**Task 15 — Interpretation**

The model starts to show overfitting roughly after the point where training accuracy continues increasing while test accuracy stops improving or starts decreasing. Use the plotted curves and results table above to identify the approximate depth for this dataset.


## 8. Model Selection, Comparison & Feature Importance

**Task 16–20**

The final model should balance test performance and the train-test gap rather than simply maximizing training accuracy.


In [ ]:
# ============================================================
# TASK 16 — SELECT A BALANCED MODEL
# ============================================================

best_row = depth_results_df.sort_values(
    by=["test_accuracy", "gap"],
    ascending=[False, True]
).iloc[0]

selected_depth = best_row["max_depth"]

if selected_depth == "Unrestricted":
    selected_depth = None

print("Selected max_depth:", selected_depth)
print("\nSelected model details:")
display(best_row.to_frame().T)


**Task 16 — Justification**

The selected depth is chosen using testing performance together with the training-testing gap. A model with strong test performance and a smaller gap is preferred because it is more likely to generalize well to unseen customers.


In [ ]:
# ============================================================
# TASK 17 — CHOSEN MODEL VS BASELINE
# ============================================================

chosen_model = DecisionTreeClassifier(
    max_depth=selected_depth,
    random_state=42
)

chosen_model.fit(X_train, y_train)

chosen_pred = chosen_model.predict(X_test)

chosen_accuracy = accuracy_score(y_test, chosen_pred)
chosen_precision = precision_score(
    y_test, chosen_pred, zero_division=0
)
chosen_recall = recall_score(
    y_test, chosen_pred, zero_division=0
)
chosen_f1 = f1_score(
    y_test, chosen_pred, zero_division=0
)

chosen_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ],
    "Chosen Model": [
        chosen_accuracy,
        chosen_precision,
        chosen_recall,
        chosen_f1
    ],
    "Baseline": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

display(chosen_results)


In [ ]:
# ============================================================
# TASK 18 — ENTROPY MODEL
# ============================================================

entropy_model = DecisionTreeClassifier(
    max_depth=selected_depth,
    criterion="entropy",
    random_state=42
)

entropy_model.fit(X_train, y_train)

entropy_pred = entropy_model.predict(X_test)

entropy_accuracy = accuracy_score(
    y_test, entropy_pred
)

entropy_precision = precision_score(
    y_test, entropy_pred, zero_division=0
)

entropy_recall = recall_score(
    y_test, entropy_pred, zero_division=0
)

entropy_f1 = f1_score(
    y_test, entropy_pred, zero_division=0
)

entropy_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ],
    "Entropy Model": [
        entropy_accuracy,
        entropy_precision,
        entropy_recall,
        entropy_f1
    ],
    "Baseline": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

display(entropy_results)


**Task 18 — Interpretation**

The entropy model is compared with the baseline to see whether changing the splitting criterion improves accuracy, precision, recall, or F1-score on the test set.


In [ ]:
# ============================================================
# TASK 19 — FEATURE IMPORTANCE
# ============================================================

importances = chosen_model.feature_importances_

feature_importance = pd.Series(
    importances,
    index=X.columns
).sort_values(ascending=False)

print("Feature Importances:")
display(feature_importance.to_frame("Importance"))

print("\nTop 5 Features:")
display(feature_importance.head(5).to_frame("Importance"))


In [ ]:
# Feature importance visualization

plt.figure(figsize=(10, 7))

feature_importance.sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Decision Tree Feature Importance")
plt.show()


## 9. Conclusion

**Task 21**

In this lab, I built a Decision Tree classifier to predict customer churn. I compared an unrestricted baseline tree with trees having different maximum depths. Controlling tree depth helps reduce overfitting and improve generalization. The final model was selected based on the balance between training and testing performance rather than training accuracy alone. Feature importance helped identify the most useful features for predicting customer churn.


## 10. Next Steps

**Task 22**

If more time were available, I would try another classification algorithm such as Random Forest or Logistic Regression and compare its performance with the Decision Tree. I would also explore feature engineering and techniques for handling class imbalance to improve churn prediction.


## 11. ID3 From Scratch — Play Badminton

**Task 23**

No built-in Decision Tree library is used in this section. Entropy and Information Gain are implemented manually, and gain values are printed at every split.


In [ ]:
# ============================================================
# PLAY BADMINTON DATASET
# ============================================================

play_data = {
    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rain",
        "Rain", "Rain", "Overcast", "Sunny",
        "Sunny", "Rain", "Sunny", "Overcast",
        "Overcast", "Rain"
    ],

    "Temperature": [
        "Hot", "Hot", "Hot", "Mild",
        "Cool", "Cool", "Cool", "Mild",
        "Cool", "Mild", "Mild", "Mild",
        "Hot", "Mild"
    ],

    "Humidity": [
        "High", "High", "High", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "High"
    ],

    "Wind": [
        "Weak", "Strong", "Weak", "Weak",
        "Weak", "Strong", "Strong", "Weak",
        "Weak", "Weak", "Strong", "Strong",
        "Weak", "Strong"
    ],

    "Play": [
        "No", "No", "Yes", "Yes",
        "Yes", "No", "Yes", "No",
        "Yes", "Yes", "Yes", "Yes",
        "Yes", "No"
    ]
}

badminton_df = pd.DataFrame(play_data)

display(badminton_df)


In [ ]:
# ============================================================
# ENTROPY FUNCTION
# ============================================================

def entropy(data, target):

    values = data[target].value_counts()
    total = len(data)

    ent = 0

    for count in values:

        p = count / total

        if p > 0:
            ent -= p * log2(p)

    return ent


print("Initial Entropy:",
      entropy(badminton_df, "Play"))


In [ ]:
# ============================================================
# INFORMATION GAIN FUNCTION
# ============================================================

def information_gain(data, attribute, target):

    total_entropy = entropy(data, target)

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[
            data[attribute] == value
        ]

        weight = len(subset) / len(data)

        weighted_entropy += (
            weight *
            entropy(subset, target)
        )

    gain = total_entropy - weighted_entropy

    return gain


In [ ]:
# ============================================================
# INITIAL INFORMATION GAINS
# ============================================================

attributes = [
    "Outlook",
    "Temperature",
    "Humidity",
    "Wind"
]

print("Initial Information Gains:")

for attribute in attributes:

    gain = information_gain(
        badminton_df,
        attribute,
        "Play"
    )

    print(
        f"{attribute}: {gain:.4f}"
    )


In [ ]:
# ============================================================
# ID3 FROM SCRATCH
# ============================================================

def id3(data, attributes, target, level=0):

    indent = "    " * level

    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    if len(attributes) == 0:
        return data[target].mode()[0]

    gains = {}

    for attribute in attributes:

        gains[attribute] = information_gain(
            data,
            attribute,
            target
        )

    print(
        f"\n{indent}Information Gain Values:"
    )

    for attribute, gain in gains.items():

        print(
            f"{indent}{attribute}: "
            f"{gain:.4f}"
        )

    # Select attribute with maximum gain
    best_attribute = max(
        gains,
        key=gains.get
    )

    print(
        f"{indent}Selected Attribute: "
        f"{best_attribute}"
    )

    tree = {
        best_attribute: {}
    }

    remaining_attributes = [
        attr
        for attr in attributes
        if attr != best_attribute
    ]

    # Create branches
    for value in data[
        best_attribute
    ].unique():

        subset = data[
            data[best_attribute] == value
        ]

        print(
            f"{indent}Branch: "
            f"{best_attribute} = {value}"
        )

        if len(subset) == 0:

            tree[
                best_attribute
            ][value] = data[
                target
            ].mode()[0]

        else:

            tree[
                best_attribute
            ][value] = id3(
                subset,
                remaining_attributes,
                target,
                level + 1
            )

    return tree


In [ ]:
# ============================================================
# BUILD ID3 TREE
# ============================================================

id3_tree = id3(
    badminton_df,
    attributes,
    "Play"
)

print("\nFinal ID3 Tree:")
print(id3_tree)


In [ ]:
# ============================================================
# PRINT TREE IN READABLE FORM
# ============================================================

def print_tree(tree, indent=""):

    # Leaf node
    if not isinstance(tree, dict):

        print(
            indent + "-> " + str(tree)
        )

        return

    attribute = list(tree.keys())[0]

    for value, subtree in tree[
        attribute
    ].items():

        print(
            indent +
            f"{attribute} = {value}"
        )

        print_tree(
            subtree,
            indent + "    "
        )


print("Readable ID3 Tree:")
print_tree(id3_tree)
